# Demo implementation of all algorithms

In [3]:
# import necessary libraries
import numpy as np
from simulate_data import simulate_views

## Simulate data

In [4]:
matrices = simulate_views(n=100, num_genes=10, M_C=10, M_Z=5, rank=3, seed=0)
print(matrices.keys())

dict_keys(['G', 'C', 'Z', 'W_C', 'W_G', 'H_G', 'H_C', 'U_G', 'U_C'])


## Run SCoNE

In [31]:
from algorithms.SCoNE import SCoNE_parallel
factor_matrices, loss_function = SCoNE_parallel(
    matrices['G'],matrices['C'],matrices['Z'], rank=3,
    alpha=max(matrices['G'].max(),matrices['C'].max())**2,lambda_H_G=0.1, lambda_H_C=0.1, lambda_Gloss=1, num_init=1,        # regularization parameters
    G_loss_type='kl_div', C_loss_type='kl_div', # in 'kl_div',  'fro' or None
)

In [32]:
print(factor_matrices.keys())

dict_keys(['W', 'H_G', 'U_G', 'H_C', 'U_C'])


In [33]:
from evaluation.reconstruction_evaluation import best_permutation_similarity

best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.729978229649069

## Run SCoNE (Fro)

In [34]:
factor_matrices, loss_function = SCoNE_parallel(
    matrices['G'],matrices['C'],matrices['Z'], rank=3,
    alpha=max(matrices['G'].max(),matrices['C'].max())**2,lambda_H_G=0.1, lambda_H_C=0.1, lambda_Gloss=1,         # regularization parameters
    G_loss_type='fro', C_loss_type='fro', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.7270225691758327

## Run CoNE

In [35]:
factor_matrices, loss_function = SCoNE_parallel(
    matrices['G'],matrices['C'],matrices['Z'], rank=3,
    alpha=0,lambda_H_G=0, lambda_H_C=0, lambda_Gloss=1,         # regularization parameters
    G_loss_type='kl_div', C_loss_type='kl_div', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.8166635754178176

## Run HNMF

In [37]:
factor_matrices, loss_function = SCoNE_parallel(
    matrices['G'],matrices['C'],None, rank=3,
    alpha=0,lambda_H_G=0, lambda_H_C=0, lambda_Gloss=1,         # regularization parameters
    G_loss_type='kl_div', C_loss_type='kl_div', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.7300014293245854

## Run HNMF (res)

In [38]:
C_resid = matrices['C'] - matrices['Z'] @ np.linalg.lstsq(matrices['Z'], matrices['C'],rcond=None)[0]
G_resid = matrices['G'] - matrices['Z'] @ np.linalg.lstsq(matrices['Z'], matrices['G'],rcond=None)[0]

factor_matrices, loss_function = SCoNE_parallel(
    G_resid,C_resid,None, rank=3,
    alpha=0,lambda_H_G=0, lambda_H_C=0, lambda_Gloss=1,         # regularization parameters
    G_loss_type='fro', C_loss_type='fro', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.653694884233639

## Run C-CoNE

In [39]:
factor_matrices, loss_function = SCoNE_parallel(
    None,matrices['C'],matrices['Z'], rank=3,
    G_loss_type=None, C_loss_type='kl_div', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.7230899121248215

## Run G-CoNE

In [40]:
factor_matrices, loss_function = SCoNE_parallel(
    matrices['G'],None,matrices['Z'], rank=3,
    G_loss_type='kl_div', C_loss_type=None, # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.6965365461269041

## Run C-NMF

In [41]:
factor_matrices, loss_function = SCoNE_parallel(
    None,matrices['C'],None, rank=3,
    G_loss_type=None, C_loss_type='kl_div', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.6948622088509918

## Run G-NMF

In [42]:
factor_matrices, loss_function = SCoNE_parallel(
    matrices['G'],None,None, rank=3,
    G_loss_type='kl_div', C_loss_type=None, # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.6649055877935475

## Run RGWAS

In [43]:
import os
from algorithms.RGWASWrapper import RGWASWrapper

cwd = os.getcwd()
parent_dir = os.path.dirname(os.getcwd())
np.save(f'{parent_dir}/example_data/G',matrices["G"])
np.save(f'{parent_dir}/example_data/C',matrices["C"])
np.save(f'{parent_dir}/example_data/Z',matrices["Z"])

factor_matrices, loss_function = RGWASWrapper(
    r_path='/gpfs/commons/home/anewbury/miniconda/bin/Rscript', # REPLACE WITH CORRECT Rscript path
    G_path=f'{parent_dir}/example_data/G.npy', C_path=f'{parent_dir}/example_data/C.npy', 
    Z_path=f'{parent_dir}/example_data/Z.npy', 
    rank=3, num_init=1)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.3704206953564072

## Run MVBC

In [44]:
from algorithms.MVBCWrapper import MVBCWrapper

factor_matrices, loss_function = MVBCWrapper(
    G_path=f'{parent_dir}/example_data/G.npy', C_path=f'{parent_dir}/example_data/C.npy', 
    rank=3, lambda_W=0.1, lambda_H_G=0.1, lambda_H_C=0.1, 
    r_path='/gpfs/commons/home/anewbury/miniconda/bin/Rscript')
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.376876663683273

In [ ]:
# per Kim and Park, set alpha to the maximum element of G or C squared
alpha = max(G.max(),C.max())**2

In [1]:
import json
import pickle

In [5]:
from utilities import deploy_run

In [6]:
deploy_run(run_name='SCoNE',out_path='',G=matrices['G'],C=matrices['C'],Z=matrices['Z'],rank=3,reg_params = {'alpha':0,'lambda_H_G':0, 'lambda_H_C':0},
           lambda_Gloss=1)

In [8]:
import pickle

with open("_factor_matrices.pkl", "rb") as f:
    factor_matrices = pickle.load(f)


with open("_loss_function.json", "r") as f:
    loss_function = json.load(f)